# English–Persian Neural Machine Translation with Transformer + FastText

This notebook implements an English-to-Persian neural machine translation pipeline using:

- the **ParsInLU English–Persian translation dataset**
- text normalization and cleaning
- **SentencePiece** tokenization
- **FastText** 300-dimensional pretrained embeddings
- a custom **Transformer encoder–decoder** implemented in TensorFlow/Keras
- BLEU and ROUGE-L evaluation
- optional BERTScore evaluation

The notebook is organized as a reproducible project workflow rather than an assignment submission.


## 1. Setup

Install the Python dependencies required by the project.


In [ ]:
%pip install -q tensorflow datasets sentencepiece scikit-learn pandas numpy matplotlib nltk rouge-score bert-score


## 2. Load Dataset

The project uses the `persiannlp/parsinlu_translation_en_fa` dataset from Hugging Face.


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "persiannlp/parsinlu_translation_en_fa",
    revision="refs/convert/parquet"
)

print(ds)
print(ds["train"][0])


## 3. Text Cleaning

English and Persian sentences are normalized separately. The preprocessing keeps basic punctuation while removing unsupported characters and repeated whitespace.


In [ ]:
import re
import unicodedata

def clean_english_text(text):
    text = text.lower().strip()
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_persian_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200c", " ")
    text = re.sub(r"[^\u0600-\u06FF?.!,]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_dataset(example):
    return {
        "en": clean_english_text(example["source"]),
        "fa": clean_persian_text(example["targets"][0]),
    }

ds_clean = ds.map(
    clean_dataset,
    remove_columns=ds["train"].column_names,
)

print(ds_clean)
print(ds_clean["train"][0])


## 4. Train SentencePiece Tokenizers

Separate unigram tokenizers are trained for English and Persian. Special token IDs are:

- PAD: `0`
- BOS: `1`
- EOS: `2`
- UNK: `3`


In [ ]:
train_data = ds_clean["train"]

with open("en_cleaned.txt", "w", encoding="utf-8") as f:
    for text in train_data["en"]:
        f.write(text + "\n")

with open("fa_cleaned.txt", "w", encoding="utf-8") as f:
    for text in train_data["fa"]:
        f.write(text + "\n")


In [ ]:
import sentencepiece as spm

VOCAB_SIZE_EN = 16000
VOCAB_SIZE_FA = 16000

spm.SentencePieceTrainer.train(
    input="en_cleaned.txt",
    model_prefix="spm_en",
    vocab_size=VOCAB_SIZE_EN,
    model_type="unigram",
    character_coverage=1.0,
    bos_id=1,
    eos_id=2,
    pad_id=0,
    unk_id=3,
)

spm.SentencePieceTrainer.train(
    input="fa_cleaned.txt",
    model_prefix="spm_fa",
    vocab_size=VOCAB_SIZE_FA,
    model_type="unigram",
    character_coverage=1.0,
    bos_id=1,
    eos_id=2,
    pad_id=0,
    unk_id=3,
)


In [ ]:
sp_en = spm.SentencePieceProcessor(model_file="spm_en.model")
sp_fa = spm.SentencePieceProcessor(model_file="spm_fa.model")

print("English:", sp_en.encode("this is a test", out_type=str))
print("Persian:", sp_fa.encode("این یک تست است", out_type=str))


## 5. Tokenization and Padding

Each sentence is wrapped with BOS/EOS tokens and padded to a fixed sequence length.


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN_EN = 50
MAX_LEN_FA = 50

def tokenize_example(example):
    return {
        "en_ids": [1] + sp_en.encode(example["en"]) + [2],
        "fa_ids": [1] + sp_fa.encode(example["fa"]) + [2],
    }

def pad_example(example):
    example["en_ids"] = pad_sequences(
        [example["en_ids"]],
        maxlen=MAX_LEN_EN,
        padding="post",
        truncating="post",
    )[0].tolist()

    example["fa_ids"] = pad_sequences(
        [example["fa_ids"]],
        maxlen=MAX_LEN_FA,
        padding="post",
        truncating="post",
    )[0].tolist()

    return example

ds_tok = ds_clean.map(tokenize_example)
ds_ready = ds_tok.map(pad_example)

print(ds_ready["train"][0])


## 6. Dataset Split

To reduce training cost, this implementation uses 20% of the original training split and divides that subset into 80% training, 10% validation, and 10% test data.


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd

full_train_df = pd.DataFrame(ds_ready["train"])

subset_df, _ = train_test_split(
    full_train_df,
    test_size=0.80,
    random_state=42,
)

train_df, temp_df = train_test_split(
    subset_df,
    test_size=0.20,
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
)

ds_train = Dataset.from_pandas(train_df, preserve_index=False)
ds_val = Dataset.from_pandas(val_df, preserve_index=False)
ds_test = Dataset.from_pandas(test_df, preserve_index=False)

print("Train:", len(ds_train))
print("Validation:", len(ds_val))
print("Test:", len(ds_test))


## 7. FastText Embeddings

Download 300-dimensional English and Persian FastText vectors and initialize the embedding matrices.

> **Implementation note:** SentencePiece produces subword pieces, while FastText vector files are primarily word based. Pieces without an exact pretrained-vector match are initialized randomly. This is an important limitation of the current implementation and a useful direction for improvement.


In [ ]:
!wget -q -O wiki-news-300d-1M.vec.zip https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
!wget -q -O cc.fa.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fa.300.vec.gz
!unzip -q -o wiki-news-300d-1M.vec.zip
!gunzip -f cc.fa.300.vec.gz

FASTTEXT_EN_PATH = "wiki-news-300d-1M.vec"
FASTTEXT_FA_PATH = "cc.fa.300.vec"
EMBED_DIM = 300


In [ ]:
import numpy as np

def load_fasttext_vec(path, dim=300):
    vectors = {}

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first = f.readline().rstrip().split()

        if len(first) != 2:
            word = first[0]
            vector = np.asarray(first[1:], dtype=np.float32)
            if vector.shape[0] == dim:
                vectors[word] = vector

        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vector = np.asarray(parts[1:], dtype=np.float32)

            if vector.shape[0] == dim:
                vectors[word] = vector

    return vectors

ft_en = load_fasttext_vec(FASTTEXT_EN_PATH, EMBED_DIM)
ft_fa = load_fasttext_vec(FASTTEXT_FA_PATH, EMBED_DIM)


In [ ]:
rng = np.random.default_rng(42)

vocab_size_en = sp_en.get_piece_size()
vocab_size_fa = sp_fa.get_piece_size()

embedding_matrix_en = np.zeros((vocab_size_en, EMBED_DIM), dtype=np.float32)
embedding_matrix_fa = np.zeros((vocab_size_fa, EMBED_DIM), dtype=np.float32)

for i in range(vocab_size_en):
    token = sp_en.id_to_piece(i)
    embedding_matrix_en[i] = ft_en.get(
        token,
        rng.normal(0, 0.05, EMBED_DIM),
    )

for i in range(vocab_size_fa):
    token = sp_fa.id_to_piece(i)
    embedding_matrix_fa[i] = ft_fa.get(
        token,
        rng.normal(0, 0.05, EMBED_DIM),
    )

print("English embedding matrix:", embedding_matrix_en.shape)
print("Persian embedding matrix:", embedding_matrix_fa.shape)


## 8. Transformer Encoder–Decoder

The translation model uses pretrained embedding matrices, sinusoidal positional encoding, multi-head self-attention, encoder–decoder cross-attention, feed-forward layers, residual connections, layer normalization, and dropout.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Embedding,
    Input,
    Dense,
    LayerNormalization,
    Dropout,
    MultiHeadAttention,
)
from tensorflow.keras.models import Model

X_enc = np.asarray(ds_train["en_ids"])
Y_full = np.asarray(ds_train["fa_ids"])

X_dec = Y_full[:, :-1]
Y = Y_full[:, 1:]

X_val_enc = np.asarray(ds_val["en_ids"])
Y_val_full = np.asarray(ds_val["fa_ids"])

X_val_dec = Y_val_full[:, :-1]
Y_val = Y_val_full[:, 1:]

MAX_LEN = 50
NUM_HEADS = 8
FF_DIM = 2048
DROPOUT = 0.1


In [ ]:
class PositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()

        pos = np.arange(max_len)[:, None]
        i = np.arange(embed_dim)[None, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / embed_dim)
        angle_rads = pos * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_emb = tf.constant(angle_rads, dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_emb[:seq_len]


class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads=8, ff_dim=2048, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(dropout)
        self.drop2 = Dropout(dropout)

    def call(self, x, training=None, mask=None):
        attn = self.attention(
            query=x,
            value=x,
            key=x,
            attention_mask=mask,
            training=training,
        )
        x = self.norm1(x + self.drop1(attn, training=training))

        ffn = self.ffn(x, training=training)
        return self.norm2(x + self.drop2(ffn, training=training))


class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads=8, ff_dim=2048, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )
        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
        )
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.norm3 = LayerNormalization(epsilon=1e-6)

        self.drop1 = Dropout(dropout)
        self.drop2 = Dropout(dropout)
        self.drop3 = Dropout(dropout)

    def call(
        self,
        x,
        encoder_output,
        training=None,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
        attn1 = self.self_attention(
            query=x,
            value=x,
            key=x,
            attention_mask=self_attention_mask,
            training=training,
        )
        x = self.norm1(x + self.drop1(attn1, training=training))

        attn2 = self.cross_attention(
            query=x,
            value=encoder_output,
            key=encoder_output,
            attention_mask=cross_attention_mask,
            training=training,
        )
        x = self.norm2(x + self.drop2(attn2, training=training))

        ffn = self.ffn(x, training=training)
        return self.norm3(x + self.drop3(ffn, training=training))


In [ ]:
embedding_encoder = Embedding(
    input_dim=vocab_size_en,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix_en],
    trainable=True,
    mask_zero=True,
    name="en_embedding",
)

embedding_decoder = Embedding(
    input_dim=vocab_size_fa,
    output_dim=EMBED_DIM,
    weights=[embedding_matrix_fa],
    trainable=True,
    mask_zero=True,
    name="fa_embedding",
)

encoder_input = Input(shape=(MAX_LEN,), dtype="int32", name="encoder_input")
decoder_input = Input(shape=(MAX_LEN - 1,), dtype="int32", name="decoder_input")

enc_emb = PositionEmbedding(MAX_LEN, EMBED_DIM)(
    embedding_encoder(encoder_input)
)

# Shape: (batch, 1, source_len)
encoder_padding_mask = tf.keras.layers.Lambda(
    lambda x: tf.cast(tf.not_equal(x, 0), tf.bool)[:, tf.newaxis, :]
)(encoder_input)

encoder_output = TransformerEncoder(
    EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
)(
    enc_emb,
    mask=encoder_padding_mask,
)

dec_emb = PositionEmbedding(MAX_LEN - 1, EMBED_DIM)(
    embedding_decoder(decoder_input)
)

# Causal decoder mask: (1, target_len, target_len)
causal_mask = tf.linalg.band_part(
    tf.ones((MAX_LEN - 1, MAX_LEN - 1), dtype=tf.bool),
    -1,
    0,
)[tf.newaxis, :, :]

# Cross-attention mask: (batch, target_len, source_len)
cross_attention_mask = tf.keras.layers.Lambda(
    lambda m: tf.tile(m, [1, MAX_LEN - 1, 1])
)(encoder_padding_mask)

decoder_output = TransformerDecoder(
    EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
)(
    dec_emb,
    encoder_output,
    self_attention_mask=causal_mask,
    cross_attention_mask=cross_attention_mask,
)

logits = Dense(vocab_size_fa, name="token_logits")(decoder_output)
model = Model([encoder_input, decoder_input], logits)

model.summary()


## 9. Training

Padding tokens are ignored in the loss function.


In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none",
)

def masked_loss(y_true, y_pred):
    token_loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, 0), token_loss.dtype)

    token_loss *= mask
    return tf.reduce_sum(token_loss) / tf.reduce_sum(mask)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    optimizer=optimizer,
    loss=masked_loss,
    metrics=["accuracy"],
)

history = model.fit(
    [X_enc, X_dec],
    Y,
    validation_data=([X_val_enc, X_val_dec], Y_val),
    epochs=10,
    batch_size=64,
)


## 10. Training Curves


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

if "accuracy" in history.history:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.show()


## 11. BLEU and ROUGE-L Evaluation

The original implementation evaluated several manually collected reference/hypothesis pairs. The same examples are retained below as qualitative samples.


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smoothing = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def compute_bleu(reference, candidate):
    return sentence_bleu(
        [reference.split()],
        candidate.split(),
        smoothing_function=smoothing,
    )

def compute_rouge(reference, candidate):
    return rouge.score(reference, candidate)["rougeL"].fmeasure

samples = [
    {
        "en": "i had my money on andy dufresne .",
        "ref": "من پولم را گذاشتم روی اندی دافرن .",
        "hyp": "من پول داشتم پول و دو تا دو تا دو تمام .",
    },
    {
        "en": "fagin , fagin ! are you a man ?",
        "ref": "فاجین ، فاجین! آیا تو انسان هستی؟",
        "hyp": "فاجین! شما مرد هستید؟",
    },
    {
        "en": "put it somewhere out of sight . where are you going .",
        "ref": "اون را دور از چشم نگه دار کجا میری",
        "hyp": "اونجا یک جایی را از دید بیرون .",
    },
    {
        "en": "tell her that dana evans she won t talk to you , miss evans .",
        "ref": "به او بگو که دانا ایوانز ایشان با شما صحبت نمی کند",
        "hyp": "خانم ایوانز: دانا ایوانز را با شما صحبت نمی کند.",
    },
    {
        "en": "you think i m no good ?",
        "ref": "خیال می کند به هیچ دردی نمی خورم؟",
        "hyp": "فکر می کنی من خوب نیستم؟",
    },
]

for i, sample in enumerate(samples, start=1):
    bleu = compute_bleu(sample["ref"], sample["hyp"])
    rouge_l = compute_rouge(sample["ref"], sample["hyp"])

    print(f"Sample {i}")
    print("EN :", sample["en"])
    print("REF:", sample["ref"])
    print("HYP:", sample["hyp"])
    print(f"BLEU: {bleu:.4f}")
    print(f"ROUGE-L: {rouge_l:.4f}")
    print("-" * 60)


## 12. Optional BERTScore

BERTScore is computationally heavier than BLEU/ROUGE. The cell below evaluates the first 1,000 test examples against their references **after model predictions have been generated**.

The current notebook does not contain a complete autoregressive decoding function, so `model_hypotheses` must first be populated with generated Persian translations.


In [ ]:
# Example usage after generating model_hypotheses:
#
# from bert_score import score
#
# references = [example["fa"] for example in ds_test.select(range(min(1000, len(ds_test))))]
# candidates = model_hypotheses[:len(references)]
#
# P, R, F1 = score(
#     candidates,
#     references,
#     lang="fa",
#     model_type="bert-base-multilingual-cased",
# )
#
# print("Average BERTScore F1:", F1.mean().item())


## Limitations and Next Steps

- SentencePiece subword tokens do not always have direct FastText word-vector matches.
- The current notebook trains on only 20% of the original training split to reduce computation.
- A complete autoregressive inference/decoding function should be added for corpus-level evaluation on the held-out test set.
- BLEU, ROUGE-L, and BERTScore should ideally be computed over model-generated translations for the full test set.
- Beam search could improve decoding quality compared with greedy decoding.
